# 01 — Enumerating the congressional hearing corpus from the GovInfo API

**Purpose.** This notebook establishes *what the corpus is* before a single
transcript is downloaded. It asks the GovInfo API for every package in the
`CHRG` collection issued between 1995 and 2025 and writes one manifest record
per package. Nothing is fetched here. The separation is deliberate: the earlier
collection behind this project interleaved discovery and retrieval, so it never
held a statement of what *should* exist against which to check what it got, and
it lost roughly 12 percent of the corpus without reporting it.

**Inputs.**

| Source | Endpoint | Contents |
|---|---|---|
| GovInfo API | `GET /published/{start}/{end}?collection=CHRG` | One record per congressional hearing package: `packageId`, `dateIssued`, `lastModified`, `congress`, `docClass`, `title`. Paginated by an `offsetMark` cursor. |

Authentication is a free API key from [api.data.gov](https://api.data.gov/signup/),
read from a `.env` in this directory. `.env` is gitignored; `.env.example` is the
template.

**Method.**

1. Query one calendar year at a time. A yearly slice keeps each cursor walk
   short, and it partitions the manifest on the same key the raw file tree
   uses, so a year can be re-collected on its own later.
2. Paginate with `offsetMark`, starting at `*` and following the `nextPage`
   cursor the API returns. The previous collector used `offset`. GovInfo orders
   `/published` by `lastModified` and re-issues packages continuously, so an
   offset-based walk reads a result set that is shifting underneath it —
   records get skipped or repeated depending on what was re-published mid-walk.
   That is the mechanism behind the two earlier collections disagreeing in both
   directions: 3,602 packages one pull never saw, 363 the other lost.
3. **Check each year's walk against the `count` the API reports.** The response
   carries the total number of matching packages on every page. If the walk
   returns a different number, `lib/govinfo.py` raises rather than returning the
   short list. This is the single most important line in the collector: it
   converts a truncated walk from silent data loss into an error at the moment
   it happens.
4. Retry transient failures — connection resets, timeouts, 429, 5xx — with
   exponential backoff, and **raise** when the budget is spent. The previous
   collector caught the same exceptions, logged them, and `break`-ed out of the
   pagination loop, handing a partial page list to a caller that treated it as
   complete. That happened 18 times.
5. Write the manifest as parquet, one file per year, plus a combined manifest.
   Parquet rather than pickle: the earlier pipeline stored everything as pickle,
   and when the filesystem zeroed 512 KiB blocks during a cluster failure, 76
   files became unreadable in a way that a columnar format would at least have
   localized.
6. Compare the result against the corpus the earlier pipeline actually ran on,
   so the size of the old gap is measured rather than asserted.

**Outputs.**

| File | Path | Contents |
|---|---|---|
| Per-year manifests | `GOVINFO_MANIFEST / packages_{year}.parquet` | One row per package, all API fields, plus the query year. |
| Combined manifest | `GOVINFO_MANIFEST / packages_all.parquet` | Every year concatenated. This is the input to `02_download_hearings.ipynb`. |
| Enumeration log | `GOVINFO_MANIFEST / enumeration_summary.csv` | Per year: reported count, records retrieved, request count. |

`GOVINFO_MANIFEST` resolves to `/scratch2/$USER/projects/llm_method/data/raw/govinfo/manifest`.
Scratch is not backed up. Everything here is reproducible from this notebook and
a valid API key; it is not archival storage.

**Notes.**

The manifest is metadata only — no transcript text — so it is small, a few MB
for the whole corpus. That is what makes it usable as a completeness contract:
it can be regenerated in a couple of minutes and diffed against a previous run
to see exactly what GovInfo has added, revised, or withdrawn.

`lastModified` on these packages is live. Sample records carry modification
timestamps well after their `dateIssued`, which is why the ordering problem in
step 2 is not hypothetical. It also means two runs of this notebook months apart
*should* differ, and the manifest is what makes that difference legible instead
of invisible.

## Setup

Paths resolve through `lib/paths.py`, so the notebook carries no project root
and no data location as a literal. The API key is read from `.env` in this
directory by `lib/govinfo.load_api_key`, which raises if it cannot find one
rather than falling back to the shared `DEMO_KEY`.

`FIRST_YEAR` and `LAST_YEAR` are declared once here and used by both the
enumeration and the verification cell, which is what keeps the reported coverage
and the persisted manifest from drifting apart. 1995 is the first year GovInfo's
`CHRG` collection covers with any density.

In [1]:
import sys

sys.path.insert(0, "../lib")

import pandas as pd

from govinfo import GovInfoClient, load_api_key
from paths import GOVINFO_MANIFEST

FIRST_YEAR = 1995
LAST_YEAR = 2025
COLLECTION = "CHRG"

GOVINFO_MANIFEST.mkdir(parents=True, exist_ok=True)

client = GovInfoClient(api_key=load_api_key(env_file=".env"))
print(f"pandas {pd.__version__}")
print(f"enumerating {COLLECTION} for {FIRST_YEAR}-{LAST_YEAR}")
print(f"manifest -> {GOVINFO_MANIFEST}")

pandas 2.3.3
enumerating CHRG for 1995-2025
manifest -> /scratch2/acroh/projects/llm_method/data/raw/govinfo/manifest


## Step 1 — Walk every year

One `offsetMark` walk per calendar year. `list_published` raises if a year's
walk returns a different number of records than the API's own `count`, so
reaching the end of this cell without an exception is itself the guarantee that
no year came back short. The per-year numbers printed below are the corpus size
by `dateIssued`, and they are the denominator every later stage is measured
against.

In [2]:
frames = []
summary = []

for year, records in client.iter_published_years(FIRST_YEAR, LAST_YEAR, COLLECTION):
    before = client.n_requests
    df = pd.DataFrame.from_records(records)
    df["query_year"] = year
    df.to_parquet(GOVINFO_MANIFEST / f"packages_{year}.parquet", index=False)
    frames.append(df)
    summary.append({"year": year, "packages": len(df), "requests": client.n_requests - before})
    print(f"{year}  {len(df):>6,} packages")

manifest = pd.concat(frames, ignore_index=True)
summary = pd.DataFrame(summary)

1995     253 packages
1996     141 packages


1997     456 packages


1998     400 packages


1999     850 packages


2000     837 packages


2001   1,125 packages


... (20 outputs left out)

2022     965 packages


2023   1,320 packages


2024     896 packages


2025     744 packages


## Step 2 — Persist the manifest

The combined manifest is the contract `02_download_hearings.ipynb` works from:
it fetches exactly the packages listed here and records an outcome for each one,
so "how many are missing" is a subtraction rather than a guess.

In [3]:
manifest = manifest.sort_values(["dateIssued", "packageId"]).reset_index(drop=True)
manifest.to_parquet(GOVINFO_MANIFEST / "packages_all.parquet", index=False)
summary.to_csv(GOVINFO_MANIFEST / "enumeration_summary.csv", index=False)

print(f"{len(manifest):,} packages, {manifest['packageId'].nunique():,} distinct ids")
print(f"dateIssued spans {manifest['dateIssued'].min()} to {manifest['dateIssued'].max()}")
print(f"{client.n_requests} API requests, {client.n_retries} retries")
print()
print(manifest.head(3).to_string(index=False))

36,378 packages, 36,378 distinct ids
dateIssued spans 1995-01-01 to 2025-12-18
53 API requests, 0 retries

congress dateIssued docClass         lastModified         packageId                                                packageLink                                        title  query_year
     104 1995-01-01     HHRG 2026-06-01T20:18:38Z CHRG-104hhrg23219 https://api.govinfo.gov/packages/CHRG-104hhrg23219/summary Oversight Hearing on the Department of Labor        1995
     104 1995-01-01     HHRG 2026-01-27T13:50:35Z CHRG-104hhrg93189 https://api.govinfo.gov/packages/CHRG-104hhrg93189/summary         CALIFORNIA SPOTTED OWL RECOVERY PLAN        1995
     104 1995-01-05     HHRG 2025-03-07T16:22:29Z CHRG-104hhrg92037 https://api.govinfo.gov/packages/CHRG-104hhrg92037/summary               Contract with America-Overview        1995


## Step 3 — By year and by chamber

`docClass` separates House (`HHRG`) from Senate (`SHRG`) hearings. Printing the
composition here is not decoration: a year whose chamber mix jumps is the first
visible symptom of a partial walk, and it is cheap to notice now rather than
after three LLM stages have run on top of it.

In [4]:
by_year = (
    manifest.pivot_table(
        index="query_year", columns="docClass", values="packageId", aggfunc="count", fill_value=0
    )
    .assign(total=lambda d: d.sum(axis=1))
)
print(by_year.to_string())

docClass    GPO  HHRG  JHRG  SHRG  Y4  Y4_AG8_1  Y4_AG8_3  Y4_C73_8  Y4_EC7  Y4_F49  Y4_F49_20  Y4_F76_1  Y4_G74_7  Y4_IN8_16  Y4_J89_1  Y4_SE2  Y4_V64_3  total
query_year                                                                                                                                                      
1995          0   243     3     7   0         0         0         0       0       0          0         0         0          0         0       0         0    253
1996          0   139     2     0   0         0         0         0       0       0          0         0         0          0         0       0         0    141
1997          0   357     2    97   0         0         0         0       0       0          0         0         0          0         0       0         0    456
1998          0   279     2   119   0         0         0         0       0       0          0         0         0          0         0       0         0    400
1999          0   574     1   275 

## Step 4 — What the earlier collection missed

The pipeline that produced this project's existing summaries, extracted topics,
and relevance scores ran on 30,032 hearings. That number is not a target — it is
the artifact of a collector with the defects described in the header — but the
difference against a clean enumeration is the honest measure of the gap, and it
belongs in the record.

In [5]:
PREVIOUS_CORPUS = 30_032  # distinct packageId in data/interim/scaleup/summarization/sum_chunks
in_window = manifest.loc[manifest["query_year"].between(1995, 2022)]

print(f"clean enumeration, 1995-2022 : {len(in_window):>7,}")
print(f"earlier pipeline corpus      : {PREVIOUS_CORPUS:>7,}")
print(f"difference                   : {len(in_window) - PREVIOUS_CORPUS:>7,} "
      f"({(len(in_window) - PREVIOUS_CORPUS) / len(in_window):.1%} of the corpus)")
print()
print(f"beyond the earlier window (2023-2025): {len(manifest) - len(in_window):,}")

clean enumeration, 1995-2022 :  33,418
earlier pipeline corpus      :  30,032
difference                   :   3,386 (10.1% of the corpus)

beyond the earlier window (2023-2025): 2,960


## Verification

Four checks run before `02_download_hearings.ipynb` is allowed to trust the
manifest. Each names the failure it catches.

1. **Distinct `packageId`.** A duplicate would mean the cursor walk revisited a
   page, which is the failure mode `offsetMark` is supposed to eliminate; if it
   appeared, every downstream count would be inflated by an unrecorded amount.
2. **Every year in range is present.** A year silently returning zero records —
   an empty `packages` list rather than an error — would otherwise look like a
   quiet year rather than a failed one.
3. **`dateIssued` falls inside the year it was queried under.** The API filters
   on `dateIssued`, so a record outside its own query window means the query
   parameters and the returned data disagree, and the yearly partition of the
   raw tree would be wrong.
4. **The fields `02` needs are non-null.** `packageId` builds every request URL
   and every output filename; a null would fail deep inside the download loop
   instead of here.

In [6]:
dupes = manifest["packageId"].duplicated().sum()
assert dupes == 0, f"{dupes} duplicate packageId — the cursor walk revisited a page"
print(f"1) {len(manifest):,} rows, all packageId distinct")

missing_years = sorted(set(range(FIRST_YEAR, LAST_YEAR + 1)) - set(manifest["query_year"]))
assert not missing_years, f"years returned no records at all: {missing_years}"
print(f"2) all {LAST_YEAR - FIRST_YEAR + 1} years present, none empty")

issued_year = pd.to_datetime(manifest["dateIssued"], format="mixed", utc=True).dt.year
off_window = int((issued_year != manifest["query_year"]).sum())
assert off_window == 0, f"{off_window} records fall outside the year they were queried under"
print("3) every dateIssued falls inside its query year")

for column in ("packageId", "dateIssued", "docClass"):
    n_null = int(manifest[column].isna().sum())
    assert n_null == 0, f"{n_null} null {column} — 02 builds URLs and filenames from these"
print("4) packageId, dateIssued, docClass all non-null")

print()
print(f"manifest ready: {GOVINFO_MANIFEST / 'packages_all.parquet'}")

1) 36,378 rows, all packageId distinct
2) all 31 years present, none empty
3) every dateIssued falls inside its query year
4) packageId, dateIssued, docClass all non-null

manifest ready: /scratch2/acroh/projects/llm_method/data/raw/govinfo/manifest/packages_all.parquet
